In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import random
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from datasets import get_dataset_config_names, load_dataset_builder
import joblib

random_state = 42

In [2]:
combined_model = joblib.load('../models/combined_tfidf_logistic_regression.joblib')

In [3]:
dataset_name = "liaad/PtBrVId"

domains = [
    "journalistic",
    "legal",
    "literature",
    "politics",
    "social_media",
    "web"
]

validation_datasets = {}

for domain in domains:
    dataset = load_dataset(
        dataset_name,
        name=domain,
        split="valid"
    )

    validation_datasets[domain] = dataset.to_pandas()

    print(f"{domain}: {len(validation_datasets[domain])} documents")

journalistic: 1000 documents
legal: 1000 documents
literature: 1000 documents
politics: 1000 documents
social_media: 1000 documents
web: 1000 documents


In [4]:
def evaluate_domain(model, df, domain):

    X = df["text"]
    y = df["label"]

    predictions = model.predict(X)

    return {
        "domain": domain,
        "accuracy": accuracy_score(y, predictions),
        "macro_f1": f1_score(
            y, predictions, average="macro"
        ),
        "classification_report": classification_report(
            y, predictions, output_dict=True
        ),
        "confusion_matrix": confusion_matrix(
            y, predictions
        )
    }

In [5]:
results = []

for domain, df in validation_datasets.items(): 
    result = evaluate_domain(
        combined_model,
        df,
        domain
    )

    results.append(result)

    print(f"{domain}: "
          f"Accuracy = {result['accuracy']:.4f}, "
          f"Macro F1 = {result['macro_f1']:.4f}")


journalistic: Accuracy = 0.9670, Macro F1 = 0.9670
legal: Accuracy = 0.7510, Macro F1 = 0.7497
literature: Accuracy = 0.5510, Macro F1 = 0.5443
politics: Accuracy = 0.8870, Macro F1 = 0.8861
social_media: Accuracy = 0.5720, Macro F1 = 0.4814
web: Accuracy = 0.1700, Macro F1 = 0.1672


In [6]:
results_df = pd.DataFrame([
    {
        "Domain": result["domain"],
        "Accuracy": result["accuracy"],
        "Macro F1": result["macro_f1"]
    }
    for result in results
])

results_df

,Domain,Accuracy,Macro F1
0,journalistic,0.967,0.967000
1,legal,0.751,0.749738
2,literature,0.551,0.544329
3,politics,0.887,0.886138
4,social_media,0.572,0.481386
5,web,0.170,0.167198


In [7]:
for result in results:

    print("\n" + "=" * 50)
    print(f"DOMAIN: {result['domain'].upper()}")
    print("=" * 50)

    print("\nConfusion Matrix:")
    print(result["confusion_matrix"])

    print("\nClassification Report:")

    report = pd.DataFrame(
        result["classification_report"]
    ).T

    print(report.round(3))


DOMAIN: JOURNALISTIC

Confusion Matrix:
[[482  18]
 [ 15 485]]

Classification Report:
              precision  recall  f1-score   support
0                 0.970   0.964     0.967   500.000
1                 0.964   0.970     0.967   500.000
accuracy          0.967   0.967     0.967     0.967
macro avg         0.967   0.967     0.967  1000.000
weighted avg      0.967   0.967     0.967  1000.000

DOMAIN: LEGAL

Confusion Matrix:
[[340 160]
 [ 89 411]]

Classification Report:
              precision  recall  f1-score   support
0                 0.793   0.680     0.732   500.000
1                 0.720   0.822     0.768   500.000
accuracy          0.751   0.751     0.751     0.751
macro avg         0.756   0.751     0.750  1000.000
weighted avg      0.756   0.751     0.750  1000.000

DOMAIN: LITERATURE

Confusion Matrix:
[[336 164]
 [285 215]]

Classification Report:
              precision  recall  f1-score   support
0                 0.541   0.672     0.599   500.000
1                

In [8]:
web_df = validation_datasets["web"].copy()

web_df["prediction"] = combined_model.predict(web_df["text"])

# Check actual and predicted label distributions
print("Actual labels:")
print(web_df["label"].value_counts())

print("\nPredicted labels:")
print(web_df["prediction"].value_counts())

Actual labels:
label
0    500
1    500
Name: count, dtype: int64

Predicted labels:
prediction
0    558
1    442
Name: count, dtype: int64


In [13]:
web_df = validation_datasets["web"].copy()

web_df["prediction"] = combined_model.predict(web_df["text"])

web_df["probability_label_1"] = (
    combined_model.predict_proba(web_df["text"])[:, 1]
)

for label in [0, 1]:

    print(f"\n{'=' * 80}")
    print(f"ACTUAL LABEL: {label}")
    print("=" * 80)

    sample = web_df[
        web_df["label"] == label
    ].sample(n=10, random_state=42)

    for _, row in sample.iterrows():

        print(f"\nActual: {row['label']}")
        print(f"Predicted: {row['prediction']}")
        print(
            f"P(label 1): "
            f"{row['probability_label_1']:.4f}"
        )
        print("-" * 80)


ACTUAL LABEL: 0

Actual: 0
Predicted: 1
P(label 1): 0.5211
--------------------------------------------------------------------------------

Actual: 0
Predicted: 0
P(label 1): 0.2175
--------------------------------------------------------------------------------

Actual: 0
Predicted: 0
P(label 1): 0.3413
--------------------------------------------------------------------------------

Actual: 0
Predicted: 0
P(label 1): 0.3398
--------------------------------------------------------------------------------

Actual: 0
Predicted: 1
P(label 1): 0.7239
--------------------------------------------------------------------------------

Actual: 0
Predicted: 1
P(label 1): 0.6754
--------------------------------------------------------------------------------

Actual: 0
Predicted: 1
P(label 1): 0.6887
--------------------------------------------------------------------------------

Actual: 0
Predicted: 1
P(label 1): 0.6121
------------------------------------------------------------------------

In [10]:
from sklearn.metrics import accuracy_score, f1_score

y_true = web_df["label"]
y_pred = web_df["prediction"]

# Original evaluation
print("Original labels:")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

# Evaluation with labels inverted
y_true_inverted = 1 - y_true

print("\nInverted labels:")
print(
    "Accuracy:",
    accuracy_score(y_true_inverted, y_pred)
)
print(
    "Macro F1:",
    f1_score(y_true_inverted, y_pred, average="macro")
)

Original labels:
Accuracy: 0.17
Macro F1: 0.16719845560465407

Inverted labels:
Accuracy: 0.83
Macro F1: 0.8294261897021581


Now we figured out that the label assignment might be reversed for web domains. 
We will also analyze whether a trained model could correctly identify Portuguese news article discussing Brazilian topic and vice versa, in order to examine whether a model has trully learned grammatical feature of both variants. 